# Problem Set 4: Transformers for gravitational wave data

Note that I will drop your lowest problem set grade out of the 4 problem sets, so if you are happy with your grade on the previous 3 problem sets, this one is optional. 

**Software requirements:** Installable via `pip` or `uv`: 
```sh
torch torchvision pillow matplotlib scikit-learn tqdm pandas seaborn wandb
```

**Dataset:** Available for download here: https://www.kaggle.com/datasets/tentotheminus9/gravity-spy-gravitational-waves

**Grading:**
This problem set will be graded as a quiz within Canvas.

**Deadline:** 
The Canvas quiz will close by end-of-day (11:59pm Central Time) on Thursday, December 18th, 2025. 

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
from tqdm.auto import tqdm
from tqdm.notebook import trange

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
### WARNING: don't change this cell -- setting these seeds will ensure that your answers are consistent with mine 
seed = 1234 
random.seed(seed)               
np.random.seed(seed)            
torch.manual_seed(seed)         
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
### set plot resolution
%config InlineBackend.figure_format = 'retina'

### set default figure parameters
plt.rcParams['figure.figsize'] = (9,6)

medium_size = 12
large_size = 15

plt.rc('font', size=medium_size)          # default text sizes
plt.rc('xtick', labelsize=medium_size)    # xtick labels
plt.rc('ytick', labelsize=medium_size)    # ytick labels
plt.rc('legend', fontsize=medium_size)    # legend
plt.rc('axes', titlesize=large_size)      # axes title
plt.rc('axes', labelsize=large_size)      # x and y labels
plt.rc('figure', titlesize=large_size)    # figure title

In [ ]:
### specify which device you'll train on: CPU, Apple GPU, or CUDA
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() # macbook
    else "cpu"
)
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Load the data

In [ ]:
### specify where the data is downloaded
folder = "./data/pset_4/"

In [ ]:
! ls $folder ### this should yield: test, train, validation, and trainingset_v1d1_metadata.csv

If the path is specified correctly, there should be 22 classes found for the train/val/test sets:

In [ ]:
print(f"Folder: {folder}")
if os.path.exists(folder):
    print("Found the data!")
    for split in ['train', 'validation', 'test']:
        split_path = os.path.join(folder, split, split)
        if os.path.exists(split_path):
            classes = os.listdir(split_path)
            print(f"  {split}: {len(classes)} classes")
else:
    print("Dataset not found.")

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 1: Class names</span>
Which of the following are actual class names?

- 1400Ripples
- Violin_Mode
- Tomato
- Air_Compressor
- Goldfish
- Woosh
- Scratchy
- Paired_Doves
- Chirp

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 2: Class counts</span>
How many examples of the `Helix` class are in the training dataset?

# Visualize the data

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 3: Different spectrograms</span>
Why are there 4 different versions of each event?
- a) Each event has the original event + 3 random augmentations
- b) Each event is shown for different stretches of time
- c) Each event has a spectrogram in the $x$, $y$, $z$, and $t$ dimensions
- d) Each event has been captured from 4 distinct gravitational wave detectors

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 4: Describing the spectrograms</span>
Try visualizing some of the images associated with each class using some code like the following:

```python
import matplotlib.pyplot as plt
from PIL import Image
fig, ax = plt.subplots(1,1)
img = Image.open("image.png")
ax.imshow(img)
```

Which of the following statements are true? 
    
- a) The Chirp events tend to be longer in duration than the Blips.
- b) The Extremely_Loud events are shorter than the Chirps in duration.
- c) The Wandering_Line events contain the same frequency at multiple distinct points in time
- d) The Whistle events start higher-frequency, then dip to lower frequencies, and then rise back up again in frequency

# Create dataloaders

In [ ]:
class GravitySpyDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, duration='1.0'):
        """
        Args:
            root_dir: Path to dataset folder
            split: 'train', 'validation', or 'test'
            transform: Optional transforms to apply
            duration: Which duration to use ('0.5', '1.0', '2.0', '4.0')
        """
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.duration = duration
        self.split_path = os.path.join(root_dir, split, split)
        self.classes = sorted([d for d in os.listdir(self.split_path) 
                               if os.path.isdir(os.path.join(self.split_path, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.idx_to_class = {idx: cls for cls, idx in self.class_to_idx.items()}
        self.samples = self._collect_samples()
        
        print(f"Loaded {len(self.samples)} samples from {split} split")
        print(f"Number of classes: {len(self.classes)}")
    
    def _collect_samples(self):
        samples = []
        
        for cls in self.classes:
            cls_path = os.path.join(self.split_path, cls)
            images = glob.glob(os.path.join(cls_path, '*.png'))
            for img_path in images:
                    if f'_{self.duration}.' in img_path or f'{self.duration}.png' in os.path.basename(img_path):
                        samples.append((img_path, self.class_to_idx[cls]))
        
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample, label = self.samples[idx]
        image = Image.open(sample).convert('RGB')
            
        if self.transform:
            image = self.transform(image)
        
        return image, label

    def get_class_weights(self):
        """Compute class weights to help account for the imbalanced data."""
        labels = [s[1] for s in self.samples]
        class_counts = Counter(labels)
        total = len(labels)
        weights = {cls: total / (len(class_counts) * count) 
                   for cls, count in class_counts.items()}
        return torch.tensor([weights[i] for i in range(len(self.classes))], dtype=torch.float32)

In [ ]:
### calculate the normalization factors 
images = [] 
for i, class_name in enumerate(all_classes):
    image = glob.glob(os.path.join(split_path, class_name, '*.png'))[0]
    img = Image.open(image)
    images.append(np.array(img))

X_train = np.vstack(np.vstack(np.array(images)))
X_train = X_train[:,:3]
X_train = X_train / 255 ### normalize
X_train = X_train.reshape(3,-1)
train_mean = X_train.mean(axis=1)
train_std = X_train.std(axis=1)
print(train_mean)
print(train_std)

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 5: Dataloaders</span>
Specify a duration of 1.0 seconds, a batch size of 32, and an image size of 128. 

How many samples are in the training split? 

In [ ]:
duration = ???
batch_size = ???
img_size = ???

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.260, 0.251, 0.245], std=[0.181, 0.165, 0.155])
])

val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.260, 0.251, 0.245], std=[0.181, 0.165, 0.155])
])

train_dataset = GravitySpyDataset(folder, split='train', transform=train_transform, duration=duration)
val_dataset = GravitySpyDataset(folder, split='validation', transform=val_transform, duration=duration)
test_dataset = GravitySpyDataset(folder, split='test', transform=val_transform, duration=duration)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(train_dataset.classes)
class_names = train_dataset.classes
class_weights = train_dataset.get_class_weights()

# Define model

In [ ]:
class PatchEmbedding(nn.Module):
    """Split image into patches and then embed them."""
    
    def __init__(self, img_size=128, patch_size=16, in_channels=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x):
        ### x: (batch_size, channels, height, width)
        x = self.projection(x)  # (batch_size, embed_dim, n_patches_h, n_patches_w)
        x = x.flatten(2)  # (batch_size, embed_dim, n_patches)
        x = x.transpose(1, 2)  # (batch_size, n_patches, embed_dim)
        return x

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 6: Multi-Head Attention</span>

In the model definition below, fill in the calculation of attention. 

What is the purpose of using `self.scale`?

- a) To normalize the output vectors to unit length
- b) To reduce memory usage during the matrix multiplication
- c) To keep dot product magnitudes consistent across different head dimensions, thereby preventing softmax saturation
- d) To make the attention weights sum to 1
- e) To ensure the query and key matrices have the same shape

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention mechanism."""
    
    def __init__(self, embed_dim=128, num_heads=4, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.projection = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x):
        batch_size, n_tokens, embed_dim = x.shape
        
        ### compute the queries, keys, and values
        qkv = self.qkv(x)  # (batch_size, n_tokens, 3*embed_dim)
        qkv = qkv.reshape(batch_size, n_tokens, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch_size, num_heads, n_tokens, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        ### TODO: Compute scaled dot-product attention using the queries (q) & keys (k) and apply to the values (v) defined above
        ### Hint: Take the dot product (pay attention to your dimensions), then apply self.scale, then apply softmax
        
        ### now calculate the attention! 
        attn = ??? # scaled dot product
        attn = ??? # softmax
        attn = self.dropout(attn)
        
        ### apply the attention to the values (v)
        x = (attn @ v).transpose(1, 2).reshape(batch_size, n_tokens, embed_dim)
        x = self.projection(x)
        x = self.dropout(x)
        
        return x

class TransformerBlock(nn.Module):
    """A single Transformer encoder block."""
    
    def __init__(self, embed_dim=128, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class ViT(nn.Module):
    """
    Vision Transformer (ViT) for Image Classification.
    """
    
    def __init__(
        self,
        img_size=128,
        patch_size=16,
        in_channels=3,
        num_classes=22,
        embed_dim=128,
        depth=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.1
    ):
        super().__init__()
        
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        
        ### learnable class token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        ### learnable position embeddings
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
        
        ### Transformer encoder blocks
        self.transformer = nn.Sequential(
            *[TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout) 
              for _ in range(depth)]
        )
        
        ### classification head
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        ### initialize weights
        self._init_weights()
    
    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        batch_size = x.shape[0]
        
        ### patch embeddings
        x = self.patch_embed(x)  # (batch_size, n_patches, embed_dim)
        
        ### adding in the class token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (batch_size, n_patches + 1, embed_dim)
        
        ### add position embeddings
        x = x + self.pos_embed
        x = self.dropout(x)
        
        ### apply the Transformer
        x = self.transformer(x)
        
        ### now do the classification task
        x = self.norm(x[:, 0]) 
        x = self.head(x)
        
        return x

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 7: Trainable parameters</span>

What is the total parameter count of our model?

In [ ]:
model = ViT(
    img_size=img_size,
    patch_size=16,
    in_channels=3,
    num_classes=num_classes,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.1
).to(device)

In [ ]:
### try it out with one example to make sure the shapes work
with torch.no_grad():
    images, labels = next(iter(train_loader))
    output = model(images.to(device))
print(f"Input shape: {images.shape}, {labels.shape}")
print(f"Output shape: {output.shape}")

# Train

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 8: Accuracy before training</span>

Fill in the code below with the `AdamW` (not `Adam`) optimizer with learning rate `1e-3`.
Define the loss function using Cross Entropy Loss with the `class_weights` defined previously.

What is the test accuracy *before* training, reported as a percent and rounded to the nearest integer?

In [ ]:
model = ViT(
    img_size=img_size,
    patch_size=16,
    in_channels=3,
    num_classes=num_classes,
    embed_dim=128,
    depth=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.1
).to(device)

optimizer = ???
loss_fn = ???

def train():
    model.train()
    for images, labels in tqdm(train_loader, desc="Training..."):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
def test(loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0
    preds = []
    trues = []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        preds.extend(pred.cpu().numpy())
        trues.extend(labels.cpu().numpy())
         
    return total_loss/len(loader.dataset), 100.*correct/total, preds, trues # loss, accuracy, preds, trues

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 9: Accuracy after training</span>

Train the model for 10 epochs and report the final test accuracy as a percent and rounded to the nearest integer.

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 10: Confusion Matrix</span>

Construct a confusion matrix (predicted class vs. true class) to investigate how accurately events of each class are correctly classified. 

For true Blip events that are misclassfied, what is the most common category that the model identifies them as?